In [1]:
%load_ext autoreload
%autoreload 2

# Preparation of proteomics data for MitoCore

This script fetches molar masses of measured proteins from Uniprot and transforms copy numbers into mmol/gDW. Based on the total protein mass, the fraction of proteins in biomass is determined as well as the mass fraction of enzymes included in the MitoCore model.

In [2]:
import cobra
import pandas

# Paths

In [17]:
proteomics_data_path = '../model_data_input/proteomics_data.xlsx'
proteomics_datasets_sorted = ['plt_proteomics']

parameterized_models_path_dict = {
    'Mitocore_Preliminary': './../parameterized_plt_models/Mitocore_Preliminary_plt.xml',
    'Mitocore_MitoMammal': './../parameterized_plt_models/Mitocore_MitoMammal_plt.xml',
    'Mitocore_aligned_to_Human1': './../parameterized_plt_models/Mitocore_aligned_to_Human1_plt.xml',
}

model_data_paths = {}
for model_name in parameterized_models_path_dict.keys():
    model_data_paths[model_name] = {
        "molar_masses": f'../model_data_input/molar_masses_{model_name}.xlsx',
        "protein_file_output": f'../model_data_input/{model_name}_protein_data',
        "mol_masses_mapping": f'../model_data_input/{model_name}_protein_id_mass_mapping.json', # rename to mol_masses_mapping
    }

# 1) Request protein molar masses from UniProt

In [4]:
from utilities.data_extraction import read_data, write_mol_weights_json

# define the columns that should be considered
columns = ['uniprot_accession', 'copies_cell']

proteomics_data_df = read_data(proteomics_data_path, proteomics_datasets_sorted, columns)

# duplicate uniprot ids correspondin to two measured peptides; values from each peptide should be averaged
# groups by uniprot_accession and then category, then averages the copies_cell values
dataset_df_mean = proteomics_data_df.groupby(['uniprot_accession', 'category'], as_index=False)['copies_cell'].mean()

In [5]:
from utilities.data_extraction import gene_id_to_uniprot_and_pw

# mapping of model gene names to uniprot ids
cobra_models = {}
gene_to_uniprot = {}

for model_name in parameterized_models_path_dict.keys():
    if model_name == "Mitocore_Original":
        continue
    cobra_models[model_name] = cobra.io.read_sbml_model(parameterized_models_path_dict[model_name])
    gene_to_uniprot_mapping = gene_id_to_uniprot_and_pw(cobra_models[model_name])
    gene_to_uniprot[model_name] = pandas.DataFrame.from_dict(gene_to_uniprot_mapping, orient='index', columns=['uniprot'])

Set parameter Username
Academic license - for non-commercial use only - expires 2026-09-26


In [19]:
from utilities.data_extraction import request_mol_weights_batch

# concat the uniprot ids from the proteomics data and the model
# request mol masses
molar_masses = {}

for model_name in parameterized_models_path_dict.keys():
    if model_name == "Mitocore_Original":
        continue
    uniprot_ids = [uniprot_id for uniprot_id in set(list(dataset_df_mean['uniprot_accession']) + list(gene_to_uniprot[model_name]['uniprot'])) if isinstance(uniprot_id, str)] # filter out non-string values
    molar_masses[model_name] = request_mol_weights_batch(uniprot_ids, output_file_name = model_data_paths[model_name]["molar_masses"])
    write_mol_weights_json(cobra_models[model_name], molar_masses[model_name], model_data_paths[model_name]["mol_masses_mapping"])

Starting retrieval of 3787 IDs in 38 batches...
Batch 1/38 completed.
Batch 2/38 completed.
Batch 3/38 completed.
Skipping malformed line in batch 4: A8MW06
Batch 4/38 completed.
Skipping malformed line in batch 5: O14582	
Skipping malformed line in batch 5: Q5BKU6
Batch 5/38 completed.
Batch 6/38 completed.
Batch 7/38 completed.
Skipping malformed line in batch 8: P08107	
Skipping malformed line in batch 8: P50224	
Skipping malformed line in batch 8: P62158	
Skipping malformed line in batch 8: Q99867
Batch 8/38 completed.
Batch 9/38 completed.
Batch 10/38 completed.
Skipping malformed line in batch 11: P0CB46
Batch 11/38 completed.
Skipping malformed line in batch 12: P0CG05
Batch 12/38 completed.
Batch 13/38 completed.
Batch 14/38 completed.
Batch 15/38 completed.
Skipping malformed line in batch 16: P0C7T9
Batch 16/38 completed.
Batch 17/38 completed.
Skipping malformed line in batch 18: Q9HCF4
Batch 18/38 completed.
Batch 19/38 completed.
Skipping malformed line in batch 20: A6NK07

# 2) Transform copies/cell to mmol/gdw, calculate total protein, calculate fraction of model included enzymes

In [20]:
from utilities.data_extraction import calculate_smoment_data_inputs

dw_per_cell = 2.13/1e12 # pg --> g

# for every condition, a targeted and global proteomics dataset should be generated
# the global datset is used to calculate the fraction of protein in biomass
# the targeted is used to calculate the fraction of model included enzymes and individual protein concentrations
for model_name in parameterized_models_path_dict.keys():
    if model_name == "Mitocore_Original":
        continue
    calculate_smoment_data_inputs(
        dataset_df_mean,
        molar_masses[model_name],
        gene_to_uniprot[model_name],
        dw_per_cell,
        model_data_paths[model_name]["protein_file_output"]
        )

No molecular weight found for A6NK07
No molecular weight found for A6NL28
No molecular weight found for A8MW06
No molecular weight found for A9Z1Y9
No molecular weight found for O14582
No molecular weight found for O15320
No molecular weight found for P01596
No molecular weight found for P01625
No molecular weight found for P08107
No molecular weight found for P0C7T9
No molecular weight found for P0CB46
No molecular weight found for P0CG05
No molecular weight found for P30042
No molecular weight found for P50224
No molecular weight found for P62158
No molecular weight found for Q13748
No molecular weight found for Q5BKU6
No molecular weight found for Q5SNT6
No molecular weight found for Q8IV90
No molecular weight found for Q99867
No molecular weight found for Q9HCF4
No molecular weight found for A6NK07
No molecular weight found for A6NL28
No molecular weight found for A8MW06
No molecular weight found for A9Z1Y9
No molecular weight found for O14582
No molecular weight found for O15320
N